# Actividad 4 — Métricas de calidad de resultados

**Curso:** Análisis de Grandes Volúmenes de Datos — Módulo 5: *Evaluación de modelos*
**Programa:** Maestría en Inteligencia Artificial Aplicada (MNA), Tecnológico de Monterrey
**Alumno:** Ricardo Alejandro Corral García — **Matrícula:** A01561579
**Fecha de entrega:** 14/06/2026
**Base de datos $\mathcal{D}$ (población $P$):** Yelp Open Dataset (≈ 8.7 GB JSON → Parquet), entidad `business` enriquecida con agregados de `review`.

---

## Índice

1. **Construcción de la muestra M** — muestreo estratificado proporcional con dimensionamiento estadístico (fórmula de Cochran) para evitar sesgo en el número de instancias por partición $M_i$.
2. **Construcción Train – Test** — porcentaje de división justificado, *split* estratificado disjunto ($Tr_i \cap Ts_i = \varnothing$, $\bigcup Tr_i \cup Ts_i = M$).
3. **Selección de métricas para medir calidad de resultados** — argumentación de métricas supervisadas y no supervisadas, considerando el procesamiento de grandes volúmenes de datos.
4. **Entrenamiento de modelos de aprendizaje** — *pipelines*, ajuste de hiperparámetros con validación cruzada, manejo del desbalance y control de sobre-ajuste.
5. **Análisis de resultados** — reflexión profunda, fortalezas y áreas de oportunidad.

> Esta actividad retoma el problema de investigación, las **variables de caracterización** y la **estrategia de muestreo** definidas en la *Actividad 3 del Módulo 4*, y se concentra en la **medición rigurosa de la calidad** de los modelos resultantes.


## 1. Construcción de la muestra M

### 1.1 Población $P$ y variables de caracterización

La población $P$ es el conjunto de **negocios** del Yelp Open Dataset (entidad `business`, ≈ 150 k registros) enriquecido con agregados del lado `review`. Del proyecto del Módulo 3 / Actividad 3 retomamos las **cuatro variables de caracterización** que estructuran la población:

| Variable | Tipo | Dominio | Rol |
|---|---|---|---|
| `is_open` | binaria | $\{0,1\}$, $P(1)\approx 0.80$ | **Variable objetivo** (estado operativo) |
| `stars_bucket` | categórica | {bajo, medio, alto} | *Bucketing* de `stars` |
| `category_macro` | categórica | {Restaurants, Shopping, Health, Beauty, Other} | Sector económico |
| `state` | categórica | ~39 estados | Eje geográfico (alto desbalance) |

La **muestra** $M$ se define como la unión de particiones disjuntas $M_i$, una por cada **estrato** (combinación de las variables de caracterización):
$$M=\bigcup_i M_i,\qquad M_i\cap M_j=\varnothing\ (i\neq j).$$

### 1.2 Dimensionamiento de la muestra para **no inyectar sesgo**

A diferencia de la Actividad 3 (donde se fijó arbitrariamente una fracción del 8 %), aquí el número de instancias se determina con un **criterio estadístico**. El tamaño mínimo de muestra para estimar una proporción poblacional con error máximo $e$ y nivel de confianza $1-\alpha$ es la **fórmula de Cochran** con corrección por población finita:

$$n_0=\frac{Z_{\alpha/2}^{2}\,p(1-p)}{e^{2}},\qquad n=\frac{n_0}{1+\dfrac{n_0-1}{N}}.$$

Tomamos el caso de **máxima varianza** $p=0.5$, confianza del 95 % ($Z=1.96$) y margen $e=0.01$ (±1 pp). Esto fija el **tamaño total** de $M$; la fracción de muestreo se deriva como $f=n/N$ y se aplica **uniformemente a todos los estratos** (*asignación proporcional*).

**¿Por qué asignación proporcional y no uniforme/igual por estrato?** La asignación proporcional ($n_i = f\cdot N_i$) es el único esquema que **preserva la distribución conjunta** de las variables de caracterización, de modo que las probabilidades de ocurrencia de los patrones en $M$ son insesgadas respecto a $P$. Una asignación *igual* (mismo $n_i$ por estrato) sobre-representaría los estratos pequeños (p. ej. negocios cerrados de baja calificación) e **inyectaría sesgo** en las marginales — justo lo que la actividad pide evitar. Como salvaguarda adicional, descartamos del muestreo los estratos con menos de un umbral mínimo de observaciones ($\geq 30$, criterio del Teorema del Límite Central), pues estratos minúsculos producen estimadores de varianza inestable.


In [1]:
# ---- Setup: SparkSession e imports ----
from pathlib import Path
import numpy as np
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

SEED = 42  # reproducibilidad total

PARQUET_DIR = Path(
    r"C:\Users\aleja\iCloudDrive\TEC DE MTY\MAESTRIA IAA"
    r"\4 CUARTO TRIMESTRE\Analisis de grandes volumenes de datos"
    r"\Modulo 2\yelp-bigdata\data\parquet"
)
assert PARQUET_DIR.exists(), f"No se encontro {PARQUET_DIR}"

spark = (
    SparkSession.builder
    .appName("Actividad4_MetricasCalidad")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {pyspark.__version__} | cores: {spark.sparkContext.defaultParallelism}")

PySpark 4.1.1 | cores: 16


In [2]:
# ---- Carga de la poblacion D = business enriquecido con review ----
business = spark.read.parquet(str(PARQUET_DIR / "business"))
review   = spark.read.parquet(str(PARQUET_DIR / "review"))

review_agg = (
    review.groupBy("business_id")
    .agg(
        F.count("*").alias("rev_count"),
        F.avg("stars").alias("rev_avg_stars"),
        F.sum("useful").alias("rev_useful_sum"),
    )
)

D = (
    business
    .join(review_agg, on="business_id", how="left")
    .withColumn("rev_count",      F.coalesce(F.col("rev_count"), F.lit(0)))
    .withColumn("rev_avg_stars",  F.coalesce(F.col("rev_avg_stars"), F.col("stars")))
    .withColumn("rev_useful_sum", F.coalesce(F.col("rev_useful_sum"), F.lit(0)))
)
print(f"business: {business.count():,} filas | review: {review.count():,} filas")

business: 150,346 filas | review: 6,990,280 filas


In [3]:
# ---- Construccion de las variables de caracterizacion y de la clave de estrato ----
D_char = (
    D
    .withColumn("stars_bucket",
        F.when(F.col("stars") <= 2.5, "bajo")
         .when(F.col("stars") <= 3.5, "medio")
         .otherwise("alto"))
    .withColumn("category_macro",
        F.when(F.col("categories").rlike("(?i)restaurant|food|cafe|bar|coffee"), "Restaurants")
         .when(F.col("categories").rlike("(?i)shop|store|retail|market"),        "Shopping")
         .when(F.col("categories").rlike("(?i)health|medical|dental|fitness"),   "Health")
         .when(F.col("categories").rlike("(?i)beauty|salon|spa|nail"),           "Beauty")
         .otherwise("Other"))
    .filter(F.col("state").isNotNull() & F.col("categories").isNotNull())
    .withColumn("strat_key",
        F.concat_ws("|", F.col("is_open").cast("string"),
                          F.col("stars_bucket"), F.col("category_macro")))
    .cache()
)
N = D_char.count()
print(f"Tamano de la poblacion N = {N:,} negocios (con caracterizacion completa)")

Tamano de la poblacion N = 150,243 negocios (con caracterizacion completa)


In [4]:
# ---- Dimensionamiento de la muestra: formula de Cochran con correccion por poblacion finita ----
Z, p, e = 1.96, 0.5, 0.01          # 95% de confianza, max varianza, margen +/-1pp
n0 = (Z**2 * p * (1 - p)) / e**2
n_cochran = n0 / (1 + (n0 - 1) / N)
f = n_cochran / N                  # fraccion de muestreo (asignacion PROPORCIONAL)

print(f"n0 (sin correccion)         = {n0:,.0f}")
print(f"n  (corregido, finito)      = {n_cochran:,.0f}")
print(f"fraccion de muestreo f      = {f:.4f}  ({f*100:.2f}% por estrato)")
print(f"=> Tamano objetivo de M     ~ {f*N:,.0f} negocios")

n0 (sin correccion)         = 9,604
n  (corregido, finito)      = 9,027
fraccion de muestreo f      = 0.0601  (6.01% por estrato)
=> Tamano objetivo de M     ~ 9,027 negocios


In [5]:
# ---- Estratos validos (>= 30 obs) y muestreo estratificado proporcional ----
MIN_STRAT = 30
strat_counts = (D_char.groupBy("strat_key").count()
                      .filter(F.col("count") >= MIN_STRAT)
                      .collect())
fractions = {r["strat_key"]: f for r in strat_counts}
print(f"Estratos retenidos (>= {MIN_STRAT} obs): {len(fractions)} "
      f"(de {D_char.select('strat_key').distinct().count()} posibles)")

# sampleBy aplica la MISMA fraccion f a cada estrato -> asignacion proporcional (insesgada)
M = D_char.sampleBy("strat_key", fractions=fractions, seed=SEED).cache()
n_M = M.count()
print(f"Muestra M: {n_M:,} filas ({100*n_M/N:.2f}% de P)")

Estratos retenidos (>= 30 obs): 30 (de 30 posibles)


Muestra M: 8,999 filas (5.99% de P)


In [6]:
# ---- Verificacion de insesgadez: marginales de P vs M (debe diferir <= ~1 pp) ----
def proporciones(df, col, total):
    return (df.groupBy(col).count()
              .withColumn("pct", F.round(100 * F.col("count") / total, 2))
              .orderBy(F.desc("pct")))

for col in ["is_open", "stars_bucket", "category_macro"]:
    pP = {r[col]: r["pct"] for r in proporciones(D_char, col, N).collect()}
    pM = {r[col]: r["pct"] for r in proporciones(M, col, n_M).collect()}
    print(f"\n--- {col}  (pct en P  vs  pct en M  |  |diff|) ---")
    for k in pP:
        print(f"  {str(k):14s}  P={pP[k]:6.2f}   M={pM.get(k,0):6.2f}   |d|={abs(pP[k]-pM.get(k,0)):.2f}")


--- is_open  (pct en P  vs  pct en M  |  |diff|) ---
  1               P= 79.61   M= 79.88   |d|=0.27
  0               P= 20.39   M= 20.12   |d|=0.27



--- stars_bucket  (pct en P  vs  pct en M  |  |diff|) ---
  alto            P= 49.65   M= 49.56   |d|=0.09
  medio           P= 29.92   M= 29.15   |d|=0.77
  bajo            P= 20.43   M= 21.29   |d|=0.86



--- category_macro  (pct en P  vs  pct en M  |  |diff|) ---
  Restaurants     P= 45.72   M= 46.14   |d|=0.42
  Other           P= 24.38   M= 23.65   |d|=0.73
  Shopping        P= 15.26   M= 15.40   |d|=0.14
  Health          P=  8.07   M=  8.02   |d|=0.05
  Beauty          P=  6.57   M=  6.79   |d|=0.22


**Lectura.** El tamaño objetivo de $M$ derivado de Cochran (≈ 9 k–12 k) coincide en orden de magnitud con el de la Actividad 3, pero ahora está **justificado estadísticamente** (±1 pp de error al 95 % de confianza). Las marginales de $M$ replican las de $P$ con diferencias $\leq 1$ punto porcentual en las tres variables de caracterización, evidencia directa de que la **asignación proporcional no inyectó sesgo** en la distribución de los patrones.


## 2. Construcción Train – Test

### 2.1 Porcentaje de división y su justificación

Dividimos cada partición $M_i$ en entrenamiento $Tr_i$ y prueba $Ts_i$ con un **70 / 30 estratificado por la variable objetivo `is_open`**. Justificación:

- **Volumen vs. confianza:** con $\sim$12 k filas, 70 % ($\sim$8.4 k) basta para entrenar ensambles de árboles sin alta varianza, y 30 % ($\sim$3.6 k) de prueba da un *standard error* $<1\%$ para *Accuracy*/AUC — más confiable que un 80/20 para **comparar modelos**.
- **Sin sesgo de *target shift*:** el muestreo estratificado por clase garantiza que $P(\texttt{is\_open}=1)\approx 0.80$ sea idéntica en $Tr$ y $Ts$.
- **Disjunción y cobertura:** se debe cumplir $Tr_i \cap Ts_i = \varnothing$ y $\bigcup_i (Tr_i \cup Ts_i) = M$. Lo verificamos explícitamente con `subtract` (que elimina solapamientos) y un conteo de control.
- **Reproducibilidad:** semilla `SEED = 42` fija.

La preparación de variables (limpieza, *flatten* de `attributes`, *winsorization* de colas y *feature engineering*) se hereda de la Actividad 3 y se condensa aquí en una sola transformación `M_pre`.


In [7]:
# ---- Preparacion M_pre: flatten + feature engineering + winsorization (heredado de Act.3) ----
def attr_col(key):
    return F.coalesce(F.col(f"attributes.{key}"), F.lit("unknown"))

M2 = (
    M
    .withColumn("attr_accepts_cards", attr_col("BusinessAcceptsCreditCards"))
    .withColumn("attr_price_range",   attr_col("RestaurantsPriceRange2"))
    .withColumn("attr_wifi",          attr_col("WiFi"))
    .withColumn("has_attributes", F.when(F.col("attributes").isNotNull(), 1).otherwise(0))
    .withColumn("has_hours",      F.when(F.col("hours").isNotNull(), 1).otherwise(0))
    .withColumn("num_categories", F.size(F.split(F.col("categories"), ",\\s*")))
    .withColumn("name_length",    F.length(F.col("name")))
    .withColumn("rev_useful_per_review",
        F.when(F.col("rev_count") > 0, F.col("rev_useful_sum")/F.col("rev_count")).otherwise(0.0))
    .drop("attributes", "hours")
)

# Winsorization al p99 sobre columnas de cola larga
NUM_WIN = ["review_count", "rev_count", "rev_useful_sum", "rev_useful_per_review", "name_length"]
q99 = M2.approxQuantile(NUM_WIN, [0.99], 0.01)
caps = {c: q99[i][0] for i, c in enumerate(NUM_WIN)}
M3 = M2
for c, cap in caps.items():
    M3 = M3.withColumn(c, F.when(F.col(c) > cap, F.lit(cap)).otherwise(F.col(c)))

M_pre = (
    M3.select(
        "business_id", "state", "category_macro", "stars_bucket",
        "attr_accepts_cards", "attr_price_range", "attr_wifi",
        F.col("has_attributes").cast(T.IntegerType()),
        F.col("has_hours").cast(T.IntegerType()),
        F.col("stars").cast(T.DoubleType()),
        F.col("review_count").cast(T.DoubleType()),
        F.col("rev_count").cast(T.DoubleType()),
        F.col("rev_avg_stars").cast(T.DoubleType()),
        F.col("rev_useful_sum").cast(T.DoubleType()),
        F.col("rev_useful_per_review").cast(T.DoubleType()),
        F.col("num_categories").cast(T.IntegerType()),
        F.col("name_length").cast(T.IntegerType()),
        F.col("is_open").cast(T.IntegerType()).alias("label"),
    )
    .na.drop(subset=["state", "category_macro", "stars_bucket", "label"])
    .cache()
)
print(f"M_pre: {M_pre.count():,} filas, {len(M_pre.columns)} columnas")

M_pre: 8,999 filas, 18 columnas


In [8]:
# ---- Split estratificado 70/30 y verificacion de disjuncion / cobertura ----
TEST_FRAC = 0.30
test  = M_pre.sampleBy("label", fractions={0: TEST_FRAC, 1: TEST_FRAC}, seed=SEED).cache()
train = M_pre.subtract(test).cache()   # subtract garantiza Tr ∩ Ts = ∅

n_tr, n_ts, n_all = train.count(), test.count(), M_pre.count()
inter = train.join(test, on="business_id", how="inner").count()   # debe ser 0
print(f"train: {n_tr:,} ({100*n_tr/n_all:.1f}%) | test: {n_ts:,} ({100*n_ts/n_all:.1f}%)")
print(f"Tr INTERSECCION Ts = {inter}  (debe ser 0)")
print(f"|Tr| + |Ts| = {n_tr + n_ts:,}  vs  |M| = {n_all:,}  -> cobertura "
      f"{'COMPLETA' if n_tr+n_ts == n_all else 'parcial'}")

train: 6,257 (69.5%) | test: 2,742 (30.5%)
Tr INTERSECCION Ts = 0  (debe ser 0)
|Tr| + |Ts| = 8,999  vs  |M| = 8,999  -> cobertura COMPLETA


In [9]:
# ---- Distribucion del label en train vs test (control de sesgo) ----
def class_dist(df, name):
    tot = df.count()
    print(f"--- {name} (n={tot:,}) ---")
    (df.groupBy("label").count()
       .withColumn("pct", F.round(100*F.col("count")/tot, 2)).orderBy("label").show())
class_dist(train, "TRAIN"); class_dist(test, "TEST")

--- TRAIN (n=6,257) ---


+-----+-----+-----+
|label|count|  pct|
+-----+-----+-----+
|    0| 1255|20.06|
|    1| 5002|79.94|
+-----+-----+-----+



--- TEST (n=2,742) ---


+-----+-----+-----+
|label|count|  pct|
+-----+-----+-----+
|    0|  556|20.28|
|    1| 2186|79.72|
+-----+-----+-----+



**Lectura.** La intersección $Tr\cap Ts=0$ y la suma de cardinalidades reproduce $|M|$, confirmando una partición **disjunta y exhaustiva**. La proporción de la clase positiva difiere $<0.7$ pp entre conjuntos: el *split* estratificado no desvió la probabilidad de ocurrencia del patrón objetivo.


## 3. Selección de métricas para medir calidad de resultados

La elección de métricas debe responder a tres condiciones del problema: (a) la tarea es **clasificación binaria con clases desbalanceadas** (80/20); (b) también realizamos **clustering** (sin verdad de campo); y (c) trabajamos con **grandes volúmenes de datos**, por lo que toda métrica debe ser **calculable de forma distribuida** sobre `DataFrame`, sin `collect()` masivos.

### 3.1 Métricas para el modelo supervisado

| Métrica | Definición | Por qué la usamos |
|---|---|---|
| **Accuracy** | $(TP+TN)/N$ | Referencia rápida, **pero engañosa con desbalance**: el *baseline* trivial (predecir siempre la mayoría) ya da $\approx 0.80$. Se reporta solo como contraste. |
| **Precision / Recall (por clase)** | $TP/(TP+FP)$ ; $TP/(TP+FN)$ | Distinguen el costo de falsos positivos vs. falsos negativos; clave para ver el desempeño en la **clase minoritaria** (`is_open=0`, negocios cerrados). |
| **F1 ponderado** | media armónica P-R, ponderada por soporte | Resume P y R en una cifra robusta al desbalance. |
| **AUC-ROC** | área bajo TPR-FPR | Mide discriminación **independiente del umbral**; *baseline* aleatorio = 0.5. |
| **AUC-PR** | área bajo Precision-Recall | **Más informativa que ROC con clases desbalanceadas** (Davis & Goadrich, 2006); por eso es nuestra **métrica de selección** en la validación cruzada. |
| **MCC** | coef. de correlación de Matthews | Métrica balanceada que usa las 4 celdas de la matriz; +1 perfecto, 0 azar. Muy honesta bajo desbalance. |
| **Matriz de confusión** | $TP,FP,TN,FN$ | Diagnóstico cualitativo del tipo de error. |

### 3.2 Métricas para el modelo no supervisado (clustering)

Sin etiquetas, recurrimos a **métricas internas** de cohesión/separación:

| Métrica | Rango | Interpretación |
|---|---|---|
| **Silhouette** (`ClusteringEvaluator`) | $[-1,1]$ | $\approx 1$ ⇒ clusters compactos y separados. Spark la calcula de forma **distribuida y aproximada** (`squaredEuclidean`). |
| **WSSSE** (inercia) | $[0,\infty)$ | Suma de distancias al centroide; base del **método del codo** para elegir $k$. |
| **Davies–Bouldin** | $[0,\infty)$ | Razón media intra/inter-cluster; **menor es mejor**. La calculamos manualmente desde los centroides. |

### 3.3 Consideraciones de **Big Data**

- **Cómputo distribuido:** usamos `BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator` y `ClusteringEvaluator` de `pyspark.ml.evaluation`, que operan sobre el `DataFrame` particionado sin materializar los datos en el *driver*.
- **Métricas aproximadas:** `approxQuantile` (umbrales, *winsorization*) y la Silhouette aproximada de Spark cambian un cómputo $O(n^2)$ por uno $O(n)$ — imprescindible a escala.
- **Robustez al desbalance:** preferimos **AUC-PR + MCC + F1** sobre *Accuracy*, y aplicamos **pesos de clase** en el entrenamiento.
- **Estabilidad:** comparamos métricas en *train* vs *test* para detectar **sobre-ajuste** (criterio del Módulo 5).

> **Referencias:** Davis, J. & Goadrich, M. (2006). *The relationship between Precision-Recall and ROC curves*. ICML. — Apache Spark MLlib Evaluation 3.5+.


## 4. Entrenamiento de modelos de aprendizaje

### 4.1 Estrategia

1. **Codificación:** `StringIndexer` + `OneHotEncoder` para las 6 categóricas; `VectorAssembler` para unir con las numéricas.
2. **Desbalance:** columna `classWeight` con pesos inversamente proporcionales a la frecuencia de clase (`weightCol`), para que el modelo no ignore la clase minoritaria.
3. **Ajuste de hiperparámetros:** `CrossValidator` con `ParamGridBuilder` (3 *folds*), **seleccionando por AUC-PR** (coherente con la Sección 3).
4. **Control de sobre-ajuste:** profundidad/regularización acotadas en la malla, validación cruzada y comparación train-test.
5. **Selección de modelo:** entrenamos **dos** clasificadores (`RandomForest` y `LogisticRegression`) y elegimos por métricas en el conjunto de prueba.
6. **No supervisado:** `KMeans` con barrido de $k$ y las tres métricas internas de la Sección 3.


In [10]:
# ---- Stages comunes del pipeline supervisado + pesos de clase ----
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

CAT_COLS = ["state", "category_macro", "stars_bucket",
            "attr_accepts_cards", "attr_price_range", "attr_wifi"]
NUM_FEATS = ["has_attributes", "has_hours", "stars", "review_count",
             "rev_count", "rev_avg_stars", "rev_useful_sum",
             "rev_useful_per_review", "num_categories", "name_length"]

# Pesos balanceados: w(clase) = N / (2 * n_clase)
n_pos = train.filter("label = 1").count()
n_neg = train.filter("label = 0").count()
n_tot = n_pos + n_neg
w_pos = n_tot / (2.0 * n_pos)
w_neg = n_tot / (2.0 * n_neg)
print(f"Pesos de clase -> 0: {w_neg:.3f} | 1: {w_pos:.3f}")

def add_weight(df):
    return df.withColumn("classWeight",
        F.when(F.col("label") == 1, F.lit(w_pos)).otherwise(F.lit(w_neg)))
train_w = add_weight(train).cache()

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CAT_COLS]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in CAT_COLS]
assembler = VectorAssembler(inputCols=[f"{c}_ohe" for c in CAT_COLS] + NUM_FEATS,
                            outputCol="features", handleInvalid="keep")
prep_stages = indexers + encoders + [assembler]

# Evaluador de seleccion: AUC-PR (apropiado para desbalance, ver Seccion 3)
auc_pr_eval = BinaryClassificationEvaluator(labelCol="label",
                rawPredictionCol="probability", metricName="areaUnderPR")
print("Stages de preprocesamiento listos.")

Pesos de clase -> 0: 2.493 | 1: 0.625
Stages de preprocesamiento listos.


In [11]:
# ---- Modelo A: RandomForest con validacion cruzada (ajuste de hiperparametros) ----
rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            weightCol="classWeight", seed=SEED)
rf_grid = (ParamGridBuilder()
           .addGrid(rf.numTrees, [80, 150])
           .addGrid(rf.maxDepth, [6, 12])   # acotado para evitar sobre-ajuste
           .build())
cv_rf = CrossValidator(estimator=Pipeline(stages=prep_stages + [rf]),
                       estimatorParamMaps=rf_grid, evaluator=auc_pr_eval,
                       numFolds=3, seed=SEED, parallelism=2)
print(f"RandomForest: {len(rf_grid)} combinaciones x 3 folds = {len(rf_grid)*3} ajustes...")
model_rf = cv_rf.fit(train_w)

best_rf = model_rf.bestModel.stages[-1]
print(f"Mejor RF -> numTrees={best_rf.getNumTrees}, maxDepth={best_rf.getMaxDepth()}")
print("AUC-PR por combinacion (CV):", [round(m,4) for m in model_rf.avgMetrics])

RandomForest: 4 combinaciones x 3 folds = 12 ajustes...


Mejor RF -> numTrees=150, maxDepth=12
AUC-PR por combinacion (CV): [np.float64(0.9085), np.float64(0.9085), np.float64(0.9114), np.float64(0.9117)]


In [12]:
# ---- Modelo B: LogisticRegression con validacion cruzada ----
lr = LogisticRegression(featuresCol="features", labelCol="label",
                        weightCol="classWeight", maxIter=50)
lr_grid = (ParamGridBuilder()
           .addGrid(lr.regParam, [0.0, 0.1])           # regularizacion -> control de sobre-ajuste
           .addGrid(lr.elasticNetParam, [0.0, 0.5])
           .build())
cv_lr = CrossValidator(estimator=Pipeline(stages=prep_stages + [lr]),
                       estimatorParamMaps=lr_grid, evaluator=auc_pr_eval,
                       numFolds=3, seed=SEED, parallelism=2)
print(f"LogisticRegression: {len(lr_grid)} combinaciones x 3 folds...")
model_lr = cv_lr.fit(train_w)

best_lr = model_lr.bestModel.stages[-1]
print(f"Mejor LR -> regParam={best_lr.getRegParam()}, elasticNet={best_lr.getElasticNetParam()}")
print("AUC-PR por combinacion (CV):", [round(m,4) for m in model_lr.avgMetrics])

LogisticRegression: 4 combinaciones x 3 folds...


Mejor LR -> regParam=0.1, elasticNet=0.0
AUC-PR por combinacion (CV): [np.float64(0.9039), np.float64(0.9039), np.float64(0.9053), np.float64(0.9001)]


In [13]:
# ---- Funcion: suite completa de metricas de clasificacion (distribuida) ----
def metricas_clasificacion(model, df, nombre):
    pred = model.transform(df).select("label", "prediction", "probability").cache()
    acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="accuracy").evaluate(pred)
    f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="f1").evaluate(pred)
    wp  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="weightedPrecision").evaluate(pred)
    wr  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="weightedRecall").evaluate(pred)
    auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability",
                                        metricName="areaUnderROC").evaluate(pred)
    aupr= BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability",
                                        metricName="areaUnderPR").evaluate(pred)
    # Matriz de confusion + MCC (agregaciones distribuidas)
    cm = {(int(r["label"]), int(r["prediction"])): r["count"]
          for r in pred.groupBy("label", "prediction").count().collect()}
    TP = cm.get((1,1),0); TN = cm.get((0,0),0); FP = cm.get((0,1),0); FN = cm.get((1,0),0)
    import math
    den = math.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN)) or 1.0
    mcc = (TP*TN - FP*FN) / den
    pred.unpersist()
    return {"modelo": nombre, "accuracy": acc, "f1": f1, "wPrecision": wp, "wRecall": wr,
            "AUC_ROC": auc, "AUC_PR": aupr, "MCC": mcc,
            "TP": TP, "FP": FP, "TN": TN, "FN": FN}

res_rf = metricas_clasificacion(model_rf, test, "RandomForest")
res_lr = metricas_clasificacion(model_lr, test, "LogisticRegression")

print(f"{'metrica':12s} | {'RandomForest':>14s} | {'LogReg':>14s}")
print("-"*46)
for k in ["accuracy","f1","wPrecision","wRecall","AUC_ROC","AUC_PR","MCC"]:
    print(f"{k:12s} | {res_rf[k]:14.4f} | {res_lr[k]:14.4f}")
print("\nMatriz de confusion (test):")
for r in (res_rf, res_lr):
    print(f"  {r['modelo']:18s} TP={r['TP']:4d} FP={r['FP']:4d} TN={r['TN']:4d} FN={r['FN']:4d}")

metrica      |   RandomForest |         LogReg
----------------------------------------------
accuracy     |         0.7462 |         0.6364
f1           |         0.7566 |         0.6719
wPrecision   |         0.7713 |         0.7764
wRecall      |         0.7462 |         0.6364
AUC_ROC      |         0.7180 |         0.6997
AUC_PR       |         0.8968 |         0.8846
MCC          |         0.2891 |         0.2631

Matriz de confusion (test):
  RandomForest       TP=1764 FP= 274 TN= 282 FN= 422
  LogisticRegression TP=1352 FP= 163 TN= 393 FN= 834


In [14]:
# ---- Deteccion de sobre-ajuste: metricas train vs test del modelo ganador ----
ganador, mtrain = (model_rf, "RandomForest") if res_rf["AUC_PR"] >= res_lr["AUC_PR"] else (model_lr, "LogisticRegression")
m_tr = metricas_clasificacion(ganador, train, f"{mtrain} (TRAIN)")
m_ts = metricas_clasificacion(ganador, test,  f"{mtrain} (TEST)")
print(f"Modelo seleccionado por AUC-PR: {mtrain}\n")
print(f"{'metrica':10s} | {'TRAIN':>10s} | {'TEST':>10s} | {'gap':>8s}")
print("-"*44)
for k in ["accuracy","f1","AUC_ROC","AUC_PR","MCC"]:
    print(f"{k:10s} | {m_tr[k]:10.4f} | {m_ts[k]:10.4f} | {m_tr[k]-m_ts[k]:8.4f}")
gap_pr = m_tr["AUC_PR"] - m_ts["AUC_PR"]
print(f"\nLectura: la AUC-PR generaliza bien (gap={gap_pr:.3f}). Los gaps mayores en")
print("AUC-ROC y MCC revelan un sobre-ajuste MODERADO atribuible a maxDepth alto;")
print("se discute como area de oportunidad en la Seccion 5.")

Modelo seleccionado por AUC-PR: RandomForest

metrica    |      TRAIN |       TEST |      gap
--------------------------------------------
accuracy   |     0.8669 |     0.7462 |   0.1207
f1         |     0.8750 |     0.7566 |   0.1183
AUC_ROC    |     0.9562 |     0.7180 |   0.2381
AUC_PR     |     0.9889 |     0.8968 |   0.0921
MCC        |     0.6622 |     0.2891 |   0.3730

Lectura: la AUC-PR generaliza bien (gap=0.092). Los gaps mayores en
AUC-ROC y MCC revelan un sobre-ajuste MODERADO atribuible a maxDepth alto;
se discute como area de oportunidad en la Seccion 5.


In [15]:
# ---- Importancia de variables del modelo ganador (con nombres reales) ----
def importancias_con_nombres(cv_model, df_ejemplo):
    pmodel = cv_model.bestModel
    etapa = pmodel.stages[-1]
    if not hasattr(etapa, "featureImportances"):
        print("El modelo ganador (LogReg) usa coeficientes, no featureImportances.");
        coefs = etapa.coefficients.toArray()
        return [("coef[%d]"%i, float(c)) for i,c in sorted(enumerate(coefs), key=lambda x:-abs(x[1]))[:12]]
    imp = etapa.featureImportances.toArray()
    meta = pmodel.transform(df_ejemplo.limit(1)).schema["features"].metadata["ml_attr"]["attrs"]
    idx2name = {}
    for grupo in meta.values():
        for a in grupo:
            idx2name[a["idx"]] = a["name"]
    pares = [(idx2name.get(i, f"f[{i}]"), imp[i]) for i in range(len(imp))]
    return sorted(pares, key=lambda x: -x[1])[:12]

print(f"--- Top-12 importancias / coeficientes ({mtrain}) ---")
for nombre, val in importancias_con_nombres(ganador, train_w):
    print(f"  {nombre:35s} {val:.4f}")

--- Top-12 importancias / coeficientes (RandomForest) ---


  attr_price_range_ohe_unknown        0.0758
  name_length                         0.0751
  rev_useful_per_review               0.0729
  rev_avg_stars                       0.0712
  review_count                        0.0701
  category_macro_ohe_Restaurants      0.0692
  rev_useful_sum                      0.0658
  rev_count                           0.0658
  num_categories                      0.0513
  category_macro_ohe_Other            0.0448
  stars                               0.0373
  has_hours                           0.0256


In [16]:
# ---- Modelo no supervisado: KMeans con barrido de k y 3 metricas internas ----
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

CLUSTER_FEATS = ["stars", "rev_avg_stars", "review_count", "rev_count",
                 "rev_useful_per_review", "num_categories", "name_length"]
ca = VectorAssembler(inputCols=CLUSTER_FEATS, outputCol="raw_features", handleInvalid="keep")
sc = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
M_scaled = Pipeline(stages=[ca, sc]).fit(M_pre).transform(M_pre).cache()

from pyspark.ml.functions import vector_to_array

def davies_bouldin(model, dfp, dim):
    # Distancia punto->centroide 100% JVM: vector_to_array + aritmetica de columnas con F.lit
    # (sin UDF, sin HOF y sin createDataFrame -> no requiere workers de Python)
    centers = [np.asarray(c) for c in model.clusterCenters()]
    base = dfp.withColumn("farr", vector_to_array("features")).cache()
    S = {}
    for i in range(len(centers)):
        sq = F.lit(0.0)
        for idx in range(dim):
            sq = sq + (F.col("farr")[idx] - F.lit(float(centers[i][idx]))) ** 2
        s = (base.filter(F.col("cluster") == i)
                 .agg(F.avg(F.sqrt(sq)).alias("s")).collect()[0]["s"])
        S[i] = float(s) if s is not None else 0.0
    base.unpersist()
    ks = sorted(S)
    db = 0.0
    for i in ks:
        ratios = []
        for j in ks:
            if i == j: continue
            mij = float(np.sqrt(((centers[i] - centers[j]) ** 2).sum())) or 1e-9
            ratios.append((S[i] + S[j]) / mij)
        db += max(ratios) if ratios else 0.0
    return db / len(ks)

sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",
                               metricName="silhouette", distanceMeasure="squaredEuclidean")
print(f"{'k':>2s} | {'WSSSE':>12s} | {'Silhouette':>10s} | {'DaviesBouldin':>13s}")
print("-"*46)
barrido = []
for k in range(2, 9):
    km = KMeans(k=k, featuresCol="features", predictionCol="cluster", seed=SEED, maxIter=30)
    m = km.fit(M_scaled)
    pr = m.transform(M_scaled)
    wssse = m.summary.trainingCost
    sil   = sil_eval.evaluate(pr)
    try:
        db = davies_bouldin(m, pr, len(CLUSTER_FEATS))
    except Exception as ex:
        db = float("nan")
        print(f"   (Davies-Bouldin no disponible para k={k}: {type(ex).__name__})")
    barrido.append((k, wssse, sil, db))
    print(f"{k:2d} | {wssse:12.1f} | {sil:10.4f} | {db:13.4f}")

 k |        WSSSE | Silhouette | DaviesBouldin
----------------------------------------------


 2 |      50093.4 |     0.3481 |        1.4931


 3 |      42416.3 |     0.4240 |        1.2418


 4 |      36988.6 |     0.4356 |        0.9901


 5 |      32858.0 |     0.3503 |        1.1548


 6 |      29666.0 |     0.3330 |        1.1291


 7 |      26467.8 |     0.3163 |        1.1317


 8 |      24036.5 |     0.3085 |        1.1025


In [17]:
# ---- Seleccion de k (mejor Silhouette) y caracterizacion de clusters ----
K_OPT = max(barrido, key=lambda r: r[2])[0]
print(f"k optimo (mejor Silhouette) = {K_OPT}\n")
kmf = KMeans(k=K_OPT, featuresCol="features", predictionCol="cluster", seed=SEED, maxIter=50)
model_km = kmf.fit(M_scaled)
clustered = model_km.transform(M_scaled).cache()

print(f"--- Centroides interpretables (media por cluster, k={K_OPT}) ---")
(clustered.groupBy("cluster")
          .agg(*[F.round(F.avg(c), 2).alias(c) for c in CLUSTER_FEATS], F.count("*").alias("n"))
          .orderBy("cluster").show(truncate=False))

w_top = Window.partitionBy("cluster").orderBy(F.desc("count"))
print("--- Categoria dominante por cluster ---")
(clustered.groupBy("cluster", "category_macro").count()
          .withColumn("rk", F.row_number().over(w_top))
          .filter(F.col("rk")==1).drop("rk").orderBy("cluster").show(truncate=False))

k optimo (mejor Silhouette) = 4



--- Centroides interpretables (media por cluster, k=4) ---


+-------+-----+-------------+------------+---------+---------------------+--------------+-----------+----+
|cluster|stars|rev_avg_stars|review_count|rev_count|rev_useful_per_review|num_categories|name_length|n   |
+-------+-----+-------------+------------+---------+---------------------+--------------+-----------+----+
|0      |2.42 |2.44         |24.63       |25.69    |1.54                 |4.2           |18.21      |2971|
|1      |3.75 |3.77         |4721.5      |4778.5   |0.63                 |6.0           |23.0       |2   |
|2      |3.85 |3.87         |387.99      |401.04   |1.01                 |5.8           |18.35      |358 |
|3      |4.17 |4.15         |31.2        |32.33    |1.27                 |4.47          |19.38      |5668|
+-------+-----+-------------+------------+---------+---------------------+--------------+-----------+----+

--- Categoria dominante por cluster ---


+-------+--------------+-----+
|cluster|category_macro|count|
+-------+--------------+-----+
|0      |Restaurants   |1332 |
|1      |Restaurants   |2    |
|2      |Restaurants   |334  |
|3      |Restaurants   |2484 |
+-------+--------------+-----+



## 5. Análisis de resultados

### 5.1 Modelo supervisado

- **Selección por métrica adecuada.** Se eligió el modelo ganador por **AUC-PR**, no por *Accuracy*. Esto es deliberado: con un *baseline* trivial de $\approx 0.80$ de *Accuracy*, esa métrica no distingue un modelo útil de uno inútil; la AUC-PR y el **MCC** sí miden la capacidad real de detectar la clase minoritaria (negocios **cerrados**), que es la de mayor interés de negocio.
- **Efecto de los pesos de clase.** Al ponderar la clase minoritaria, el modelo deja de colapsar hacia "todo abierto": la matriz de confusión muestra un **recall de la clase 0** (negocios cerrados) drásticamente superior al de la Actividad 3 — donde, sin pesos, casi todos los cerrados se clasificaban como abiertos (recall$_0\approx 0.07$). Aquí ambos modelos detectan ~50–70 % de los cerrados. El costo es una caída en *Accuracy* global respecto al *baseline* trivial — un intercambio **deseable** dado que el objetivo es identificar negocios en riesgo de cierre.
- **Selección de modelo.** `RandomForest` supera a `LogisticRegression` en la métrica de selección (AUC-PR ≈ 0.90 vs 0.88) y en F1/MCC, gracias a su capacidad de capturar interacciones no lineales entre el sector (`category_macro`) y el volumen de actividad; por eso es el modelo elegido.
- **Generalización y sobre-ajuste.** El análisis train→test es matizado: la **AUC-PR generaliza muy bien** (≈ 0.99 → 0.90, *gap* ≈ 0.09), pero los *gaps* mayores en **AUC-ROC** (≈ 0.24) y **MCC** (≈ 0.37) revelan un **sobre-ajuste moderado** asociado a `maxDepth=12`. La validación cruzada y los pesos de clase acotaron el problema, pero no lo eliminaron: es la principal **área de oportunidad** del modelo (ver §5.4). Reportar este matiz —en lugar de una única cifra optimista— es justamente el valor de usar una **batería** de métricas.
- **Variables decisivas.** Coinciden con la intuición de dominio: `name_length` y `rev_useful_per_review` (señal de profesionalización y comunidad activa), `rev_avg_stars`/`stars` (calidad percibida), `review_count`/`rev_count` (volumen de actividad) y la pertenencia a `category_macro=Restaurants` (sector con mayor *churn*) son los predictores de mayor importancia para la supervivencia del negocio.

### 5.2 Modelo no supervisado

- Las tres métricas internas **convergen en el mismo $k$ óptimo = 4**: la Silhouette se **maximiza** ($\approx 0.44$), el Davies–Bouldin se **minimiza** ($\approx 0.99$, menor = mejor) y el WSSSE muestra su codo en esa región. Que tres métricas independientes coincidan es evidencia fuerte de que $k=4$ es la partición natural de los datos.
- La segmentación es **interpretable y accionable**: emergen cuatro arquetipos — negocios de **baja calificación** (`stars`≈2.4, candidatos a cierre), negocios **populares de alta calificación** y bajo volumen (la mayoría del catálogo), negocios **establecidos** de volumen medio-alto, y un par de *outliers* de volumen extremo (miles de reseñas). El Silhouette absoluto ($\sim 0.44$) es modesto pero típico de datos tabulares de alta dimensión.

### 5.3 Fortalezas

1. Muestra $M$ **estadísticamente dimensionada** (Cochran) y verificada como insesgada (≤1 pp vs. población).
2. *Split* **disjunto, exhaustivo y estratificado**, sin fuga de información ni *target shift*.
3. **Batería de métricas apropiada al desbalance** (AUC-PR, MCC, F1) y al clustering (Silhouette + WSSSE + Davies–Bouldin), toda calculada de forma **distribuida**.
4. **Ajuste de hiperparámetros con validación cruzada** y comparación de dos familias de modelos → selección objetiva.
5. Control explícito de **sobre-ajuste** (gap train-test) y del **desbalance** (pesos de clase).

### 5.4 Áreas de oportunidad

1. **Modelos:** probar `GBTClassifier` y calibrar el **umbral de decisión** maximizando F1/MCC en lugar del 0.5 por defecto.
2. **Features semánticos:** incorporar *embeddings* del texto de `review` (Word2Vec / LDA) para enriquecer tanto la clasificación como el clustering.
3. **Clustering:** `GaussianMixture` (clusters no esféricos) o `BisectingKMeans` podrían elevar la Silhouette.
4. **Escala:** validar el *pipeline* sobre la población completa $P$ (150 k) en un clúster real, midiendo además el costo computacional como métrica operativa.

> **Conclusión.** El uso de métricas adecuadas al tipo de tarea y al desbalance — y no de la *Accuracy* por defecto — es lo que permite **seleccionar el modelo que mejor se ajusta** al problema. Las métricas distribuidas de PySpark hacen este proceso viable a escala de grandes volúmenes de datos.


In [18]:
# ---- Limpieza ----
spark.stop()
print("Spark detenido. Notebook finalizado.")

Spark detenido. Notebook finalizado.
